In [ ]:
import math
import torch.nn.functional as F
import timeit
from typing import Iterable
import torch
from torch import nn
from einops import rearrange, einsum, reduce
from edtrace import text, image, link
from lecture_util import article_link
from gpu_util import cuda_if_available, get_max_memory_usage
from facts import h100_flop_per_sec, h100_bytes_per_sec
from references import deepseek_v3_2_2025, adagrad_2011, nemotron_3_super_2026


# CS336: 从头开始构建语言模型 (2026春季)

# 第二讲：资源核算 (Resource Accounting)


### 课程公告

- 加入 CS336 Slack 频道

- 使用你的 **Stanford** 邮箱注册并加入 Modal 算力平台

- 阅读 [AI 政策指南](https://docs.google.com/document/d/1SZAlExB1qAc9izHt54gwunNpjKE6wXb8Y7yA_e-baK8/edit?tab=t.0)

- 阅读 [集群使用指南](https://docs.google.com/document/d/1cHE0iKVyXLJ3XpIs2XuXTmZ-HMmPk2hIPeCvy-AydMg/edit?tab=t.otis27tacaef)

Marin $10^{23}$ FLOPs 的模型预训练已顺利完成，并且[非常完美地符合我们的预测损失](https://x.com/WilliamBarrHeld/status/2039373983632814318)！

<img src="https://pbs.twimg.com/media/HE1P1HmaUAAjLXF?format=jpg&name=medium" width="800" />

上一讲内容回顾：课程概述与分词 (Tokenization)。

今天主题：**系统底座的资源核算 (Resource Accounting)**。

> **核心出发点**：在硬件资源（算力、显存）固定的情况下，如何训练出最好的模型？

> 也就是说，要最大化**计算效率**。

> **前提条件**：准确分析和核算给定计算任务的资源消耗。


### 辅助工具函数 (Utility Helper Functions)

在正式开始本节课的实验前，我们先定义一些底层的辅助工具函数（计算张量显存大小、查询 GPU 理论峰值 FLOPS、基准测试测速套件等）。这些辅助函数将在后续各章节中被直接调用。

In [ ]:
def get_memory_usage(x: torch.Tensor):
    return x.numel() * x.element_size()


In [ ]:
def get_promised_flop_per_sec(dtype: torch.dtype) -> float:
    """Return the peak FLOP/s for `device` operating on `dtype`."""
    if not torch.cuda.is_available():
        # 无 CUDA 设备可用，返回 1
        return 1
    properties = torch.cuda.get_device_properties(cuda_if_available())

    if "A100" in properties.name:
        # https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/nvidia-a100-datasheet-us-nvidia-1758950-r4-web.pdf
        if dtype == torch.float32:
            return 19.5e12
        if dtype in (torch.bfloat16, torch.float16):
            return 312e12
        raise ValueError(f"Unknown dtype: {dtype}")

    if "H100" in properties.name:
        # https://www.nvidia.com/en-us/data-center/h100/
        if dtype == torch.float32:
            return 67.5e12
        if dtype in (torch.bfloat16, torch.float16):
            return 1979e12 / 2  # 1979 is for sparse, dense is half of that
        raise ValueError(f"Unknown dtype: {dtype}")

    if "B200" in properties.name:
        # https://www.primeline-solutions.com/media/categories/server/nach-gpu/nvidia-hgx-h200/nvidia-blackwell-b200-datasheet.pdf
        if dtype == torch.float32:
            return 75e12
        if dtype in (torch.bfloat16, torch.float16):
            return 4.5e15 / 2  # 4.5e15 is for sparse, dense is half of that
        raise ValueError(f"Unknown dtype: {dtype}")

    # Unknown GPU: return None so caller can handle gracefully
    return None


In [ ]:
def benchmark(func, num_trials: int = 5) -> float:
    """Return the number of seconds required to perform `func`."""

    # 同步等待之前的 CUDA 线程执行完毕
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    def run():
        # Perform the operation
        func()

        # Wait until CUDA threads are done
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    # Time the operation `num_trials` times
    total_time = timeit.timeit(run, number=num_trials)

    return total_time / num_trials


In [ ]:
def get_num_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters())


1. **问题**：在 1024 张 H100 GPU 上，训练一个 70B 参数的模型（在 15T Token 上预训练）需要多久？


In [ ]:
total_flops = 6 * 70e9 * 15e12
h100_flop_per_sec = 1979e12 / 2
mfu = 0.5
flops_per_day = h100_flop_per_sec * mfu * 1024 * 60 * 60 * 24
days = total_flops / flops_per_day


2. **问题**：在 8 张 H100 (80GB) GPU 上，使用 AdamW 优化器，能训练的最大模型参数量是多少？


In [ ]:
h100_bytes = 80e9
bytes_per_parameter = 2 + 2 + (4 + 4)  # 参数(2), 梯度(2), 优化器状态(4 + 4)
num_parameters = (h100_bytes * 8) / bytes_per_parameter


说明：这里未计算激活值显存（它取决于批大小和序列长度），因此这仅是模型参数量的上限。

这是一个非常粗略的估算（Napkin Math）。

但它能让你体会到通过物理账本快速估算资源占用和训练耗时的方法。

本讲的核心知识：

- **机制 (Mechanics)**：基础操作（PyTorch 语法）

- **心态 (Mindset)**：学会进行资源核算，凡事量化分析

- **直觉 (Intuitions)**：对资源如何被消耗有一个大致的概念（这里没有 ML 魔法，只有 Napkin Math 物理账本）


张量（Tensor）是存储一切的底层基本构建模块：

- 数据 (Data)

- 参数 (Parameters)

- 梯度 (Gradients)

- 优化器状态 (Optimizer state)

- 激活值 (Activations)

例如：DeepSeek v3.2 模型的参数。[DeepSeek-V3.2 (DeepSeek-AI, 2025)](https://arxiv.org/abs/2512.02556)

[DeepSeek v3.2 model on Hugging Face](https://huggingface.co/deepseek-ai/DeepSeek-V3.2?show_file_info=model.safetensors.index.json)

每个张量都有一个秩（Rank），即它的维度数量。


In [ ]:
x = torch.zeros(4)        # 秩为 1 的张量（向量）
x = torch.zeros(4, 8)     # 秩为 2 的张量（矩阵）
x = torch.zeros(4, 8, 2)  # 秩为 3 的张量


在 Transformer 中，我们经常会看到秩为 4 的张量：


In [ ]:
B = 32   # 批大小
S = 16   # 序列长度
H = 16   # 注意力头数
D = 64   # 每个头的隐藏层维度
x = torch.zeros(B, S, H, D)


张量的元素通常是浮点数。

## fp32 (单精度)

[Wikipedia](https://en.wikipedia.org/wiki/Single-precision_floating-point_format)

<img src="images/fp32.png" width="700" />

fp32 数据类型（也称为 float32 或单精度）是默认的格式。

传统上，在科学计算中，fp32 是基线，甚至在某些情况下会使用双精度（fp64）。

但在深度学习中，我们可以对精度“粗心”得多。

让我们看看这些张量的显存占用情况。

显存占用是由（1）数值的个数 和（2）每个数值的数据类型 共同决定的。


In [ ]:
x = torch.zeros(4, 8)
assert x.dtype == torch.float32  # 默认数据类型
assert x.numel() == 4 * 8
assert x.element_size() == 4  # float32 占用 4 字节
assert get_memory_usage(x) == 4 * 8 * 4  # 128 bytes


GPT-3 前馈层（FFN）中的一个矩阵的大小：


In [ ]:
assert get_memory_usage(torch.empty(12288 * 4, 12288)) == 2304 * 1024 * 1024  # 2.3 GB


## fp16 (半精度)

[Wikipedia](https://en.wikipedia.org/wiki/Half-precision_floating-point_format)

<img src="images/fp16.png" width="400" />

fp16 数据类型（也称为 float16 或半精度）可以将内存减半。


In [ ]:
x = torch.zeros(4, 8, dtype=torch.float16)
assert x.element_size() == 2


然而，fp16 的动态范围（尤其是针对极小数值）并不够大。


In [ ]:
x = torch.tensor([1e-8], dtype=torch.float16)
assert x == 0  # 数值下溢！


如果在训练过程中发生这种情况，很容易导致数值不稳定（训练崩溃）。

## bf16 (Bfloat16)

[Wikipedia](https://en.wikipedia.org/wiki/Bfloat16_floating-point_format)

<img src="images/bf16.png" width="400" />

Google Brain 研发的 bfloat16 格式，每个数值占用 2 字节。它具有与 fp32 相同的动态范围，但尾数精度较低。在大型模型训练中，bf16 可以免去 fp16 易发生下溢的梯度缩放操作。

bf16 与 fp16 占用相同的内存，但具有与 fp32 相同的动态范围！

唯一的妥协是其精度分辨率稍差，但这在深度学习中往往不是问题。


In [ ]:
x = torch.tensor([1e-8], dtype=torch.bfloat16)
assert x != 0  # 未发生下溢！


## 混合精度 (Mixed precision)

这对训练的影响：

- 用 fp32 训练最稳定，但需要非常庞大的显存。

- 全程使用 fp16 甚至 bf16 存在极高的不稳定性风险。

解决方案：混合精度训练 (Mixed precision training)[https://arxiv.org/pdf/1710.03740.pdf](https://arxiv.org/pdf/1710.03740.pdf)

- 参数、激活值和梯度使用 bf16 存储与计算

- 优化器状态（一阶、二阶动量）使用 fp32 存储与累加

PyTorch 提供了自动混合精度 (AMP) 库。[docs](https://pytorch.org/docs/stable/amp.html)

它会在安全的情况下自动将操作（例如矩阵乘法而非指数操作）转换为 bf16 计算。


In [ ]:
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
    x = torch.zeros(4, 8)


## fp8

2022年，受机器学习工作负载的推动，fp8 得到了标准化。

<img src="https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/_images/fp8_formats.png" width="600" />

H100 GPU 支持两种 FP8 变体：E4M3（动态范围 [-448, 448]）和 E5M2（动态范围 [-57344, 57344]）。

参考资料：[https://arxiv.org/pdf/2209.05433.pdf](https://arxiv.org/pdf/2209.05433.pdf)

## fp4

2025年，NVIDIA 推出了针对超高效率推理的 [nvfp4](https://developer.nvidia.com/blog/introducing-nvfp4-for-efficient-and-accurate-low-precision-inference/)。

每个值仅占用 4 比特！

可选数值：-6, -4, -3, -2, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2, 3, 4, 6

在每个块内使用独立的缩放因子，从而在小范围内获得更高的动态范围（但无法脱离块内邻居随意变化）。

Nemotron 3 Super 是使用 NVFP4 进行训练的。[Nemotron 3 Super: Open, Efficient Mixture-of-Experts Hybrid Mamba-Transformer Model for Agentic Reasoning (NVIDIA, 2026)](https://research.nvidia.com/labs/nemotron/files/NVIDIA-Nemotron-3-Super-Technical-Report.pdf)

其中许多底层的量化操作都是由 NVIDIA 底层库在后台完成的，对普通用户是不透明的。


默认情况下，张量是存储在 CPU 内存中的。


In [ ]:
x = torch.zeros(32, 32)
assert x.device == torch.device("cpu")


然而，对于 GPU 呢？

<img src="images/cpu-gpu.png" width="600" />


In [ ]:
device = cuda_if_available()


为了利用 GPU 庞大的并行计算能力，我们需要将它们移动至 GPU 显存中。


In [ ]:
x = x.to(device)


或者直接在 GPU 上创建张量：


In [ ]:
with torch.device(device):
    x = torch.zeros(32, 32)
    assert x.device == device


传统的 PyTorch 代码：


In [ ]:
x = torch.ones(2, 2, 3)      # 批大小 序列长度 隐藏维度
y = torch.ones(2, 2, 3)      # 批大小 序列长度 隐藏维度
z = x @ y.transpose(-2, -1)  # 批大小 序列长度 序列长度


传统的写法非常容易搞错维度（到底 -2 和 -1 代表什么维度？）……

Einops 是一个极其强大且直观的张量操作库，允许我们直接给维度命名。

它的设计灵感来源于爱因斯坦求和约定 (Einstein, 1916)。

[Einops tutorial](https://einops.rocks/1-einops-basics/)


Einsum 是包含维度簿记的泛化矩阵乘法。


In [ ]:
x = torch.ones(3, 4)  # 序列1 隐藏维度
y = torch.ones(4, 3)  # 隐藏维度 序列2
z = x @ y   # seq1 seq2
z = einsum(x, y, "seq1 hidden, hidden seq2 -> seq1 seq2")


让我们来看一个更复杂的例子……


In [ ]:
x = torch.ones(2, 3, 4)  # 批大小 序列1 隐藏维度
y = torch.ones(2, 3, 4)  # 批大小 序列2 隐藏维度
z = x @ y.transpose(-2, -1)  # batch seq1 seq2
z = einsum(x, y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2")


在输出部分未被命名的维度，将在计算时被自动求和缩并。


In [ ]:
z = einsum(x, y, "... seq1 hidden, ... seq2 hidden -> ... seq1 seq2")


你可以通过某些操作（如求和、均值、最大值、最小值）来对单个张量的某些维度进行规约（Reduce）。


In [ ]:
x = torch.ones(2, 3, 4)  # 批大小 序列长度 隐藏维度
y = x.sum(dim=-1)
y = reduce(x, "... hidden -> ...", "sum")


有时，一个物理维度实际上代表了两个逻辑维度。

……而你希望只对其中一个维度进行操作。


In [ ]:
x = torch.ones(3, 8)  # 序列 隐藏维度总和


……其中 `total_hidden` 是 `heads * hidden1` 平铺展开后的表示。


In [ ]:
w = torch.ones(4, 4)  # 隐藏维度1 隐藏维度2
x = rearrange(x, "... (heads hidden1) -> ... heads hidden1", heads=2)
x = einsum(x, w, "... hidden1, hidden1 hidden2 -> ... hidden2")
x = rearrange(x, "... heads hidden2 -> ... (heads hidden2)")


了解了这些基础操作后，接下来让我们深入核算它们的计算开销（FLOPs）。

浮点运算次数 (FLOP) 是指最基础的浮点数学运算（如一次加法 $x+y$ 或一次乘法 $x \cdot y$）。

两个非常容易混淆的英文缩写（读音完全相同）：

- **FLOPs**：浮点运算次数，是衡量执行的**总计算量**的度量。

- **FLOP/s**（或 FLOPS）：每秒浮点运算次数，用于衡量**硬件计算速度**的性能指标。

## 直觉感受

训练 GPT-3 (2020) 消耗了约 $3.14 \times 10^{23}$ FLOPs 的总计算量。 [相关文章](https://lambdalabs.com/blog/demystifying-gpt-3)

据估算，训练 GPT-4 (2023) 消耗了约 $2 \times 10^{25}$ FLOPs。 [相关文章](https://patmcguinness.substack.com/p/gpt-4-details-revealed)

H100 标称的半精度稀疏峰值性能为 1979 TFLOPS，无稀疏时减半。[spec](https://resources.nvidia.com/en-us-tensor-core/nvidia-tensor-core-gpu-datasheet)


In [ ]:
h100_flop_per_sec = 1979e12 / 2


用 8 张 H100 GPU 连续训练两星期能提供的最大算力容量：


In [ ]:
total_flops = 8 * 2 * (60 * 60 * 24 * 7) * h100_flop_per_sec


## 线性模型 (Linear Model)


In [ ]:
if torch.cuda.is_available():
    B = 16384  # 样本数量
    D = 32768  # 样本维度
    K = 8192   # 输出维度
else:
    B = 1024
    D = 256
    K = 64
x = torch.ones(B, D, device=cuda_if_available())
w = torch.randn(D, K, device=cuda_if_available())
y = x @ w


该矩阵乘法（matmul）共包含多少次浮点运算？

对于每一个 $(i, j, k)$ 元组，我们需要执行一次乘法 and 一次加法。


In [ ]:
actual_num_flops = 2 * B * D * K


我们也可以对这个操作进行测速。


In [ ]:
actual_time = benchmark(lambda: x @ w)


该操作的实际每秒浮点运算速度 (FLOP/s)：


In [ ]:
actual_flop_per_sec = actual_num_flops / actual_time


每款 GPU 都有对应的规格表来提供它的峰值理论性能指标。

- 例如：[H100 spec](https://resources.nvidia.com/en-us-gpu-resources/h100-datasheet-24306)

需要注意的是，FLOP/s 在极大程度上取决于所使用的数据类型精度！


In [ ]:
promised_flop_per_sec = get_promised_flop_per_sec(x.dtype)


## 模型算力利用率 (Model FLOPs Utilization, MFU)

定义：$\text{MFU} = \frac{\text{实际吞吐 FLOP/s}}{\text{硬件峰值 FLOP/s}}$（不含通信和框架冗余）。


In [ ]:
mfu = actual_flop_per_sec / promised_flop_per_sec if promised_flop_per_sec else None


在实际大规模模型预训练中，MFU $\ge 0.5$ 就已经是非常不错的硬件利用效率了！

为什么 MFU 难以接近 1.0？

为了回答这个问题，我们需要更深入地了解数据是如何在 GPU 的硬件层面上流转和计算的……


<img src="images/compute-memory.png" width="300" />

硬件执行一次计算的基本步骤：

1. 将输入数据从全局显存 (HBM) 加载至计算核心 (Core / Register)

2. 核心执行实际的浮点计算

3. 将计算完成的输出写回全局显存 (HBM)

整个过程需要多少时间？

这取决于两个关键的硬件指标：

1. 运算核心的速度 (FLOP/s)

2. 全局显存的带宽 (Bytes/s)


In [ ]:
assert h100_flop_per_sec == 1979e12 / 2  # Half without sparsity
assert h100_bytes_per_sec == 3.35e12


In [ ]:
n = 1024 * 1024
x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())
y = torch.relu(x)
bytes = (2 * n) + (2 * n)  # 读取 x, 写入 y (bf16 为 2 字节)
flops = n  # n 次比较
communication_time = bytes / h100_bytes_per_sec
computation_time = flops / h100_flop_per_sec


假设我们能将数据传输与浮点计算完美地重叠进行。


In [ ]:
total_time = max(communication_time, computation_time)


什么是瓶颈所在？

- **显存带宽受限 (Memory-bound)**：传输数据所花的时间长于核心计算的时间。

- **算力受限 (Compute-bound)**：核心计算的时间长于数据传输的时间。

在这种情况下，单独的 ReLU 操作显然是显存带宽受限（Memory-bound）的。

另一种等价的分辨方法：

硬件计算强度 (Accelerator Intensity)：硬件每秒传输一个字节的数据，计算核心理论上能做多少次计算？


In [ ]:
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec


算法的算术强度 (Arithmetic Intensity)：当前算子中，平均每传输一个字节的数据，实际执行了多少次浮点运算？


In [ ]:
arithmetic_intensity = flops / bytes  # ~1/4


什么是瓶颈所在？

- Memory-bound: arithmetic intensity < accelerator intensity

- Compute-bound: arithmetic intensity > accelerator intensity


In [ ]:
assert arithmetic_intensity < h100_accelerator_intensity


你会发现，在许多元素级操作中，我们都处于显存受限（Memory-bound）状态。

我们能设法提高算术强度吗？


In [ ]:
n = 1024 * 1024
x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())
y = F.gelu(x)  # GELU(x) = 0.5 x (1 + tanh(sqrt(2/pi) (x + 0.044715 x^3)))
bytes = (2 * n) + (2 * n)  # 读取 x, 写入 y (bf16 为 2 字节)
flops = 20 * n  # tanh 可以通过多项式等多种方式进行近似
arithmetic_intensity = flops / bytes
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec
assert arithmetic_intensity < h100_accelerator_intensity


我们注意到，由于 GeLU 包含复杂的 tanh/立方项计算，它在每移动一个字节的数据时执行了更多计算，因而其算术强度要显著高于 ReLU。

但它依然处于显存受限阶段！

换言之，单独执行时，ReLU 并不比 GeLU 跑得更快（因为瓶颈全都在读写显存上）。


In [ ]:
n = 1024 * 1024
x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())
w = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())
y = x @ w
bytes = (2 * n) + (2 * n) + 2  # 读取 x, 读取 w, 写入 y
flops = 2 * n - 1  # n 次乘法，n-1 次加法
arithmetic_intensity = flops / bytes  # ~1/2
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec
assert arithmetic_intensity < h100_accelerator_intensity


显存带宽受限！


In [ ]:
n = 1024
x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())
w = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())
y = x @ w
bytes = (2 * n) + (2 * n * n) + (2 * n)  # 读取 x, 读取 w, 写入 y
flops = n * (2 * n - 1)  # n 次点积
arithmetic_intensity = flops / bytes  # ~1
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec
assert arithmetic_intensity < h100_accelerator_intensity


显存带宽受限！


In [ ]:
n = 1024
x = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())
w = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())
y = x @ w
bytes = (2 * n * n) + (2 * n * n) + (2 * n * n)  # 读取 x, 读取 w, 写入 y
flops = n * n * (2 * n - 1)  # n^2 次点积
arithmetic_intensity = flops / bytes  # ~n/3
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec
assert arithmetic_intensity > h100_accelerator_intensity


终于，进入算力受限（Compute-bound）状态！

只要矩阵维度足够大，乘法操作就能彻底掩盖显存传输开销，进入算力受限状态（榨干 GPU 算力）。

这就是为什么在训练 Transformer 时，我们主要处于算力受限（矩阵乘法为主），这很有利于压榨硬件性能。

而在大模型推理（生成 Token）时，由于一次只处理一个 Token，矩阵乘法退化为矩阵-向量乘法，这也正是为什么大模型推理极度依赖显存带宽。

注：算术强度与硬件计算强度的权衡，同样也高度依赖于我们选用的数值精度类型（例如 bf16 的硬件计算强度明显高于 fp32）。


我们可以利用 Roofline 模型（屋顶图）非常直观地展现算法算术强度与硬件实际性能之间的关系。

<img src="https://jax-ml.github.io/scaling-book/assets/img/roofline-improved-1400.webp" width="600" />

- 横坐标 $x$ 代表算法的算术强度（每字节传输对应的计算次数）

- 折线图代表特定硬件在当前算术强度下能发挥的最大实际性能

- 折弯处的拐点（Kink）正是硬件计算强度（标志着从显存受限向算力受限的过渡）

此时，我们将这与 MFU 关联起来：

MFU = min(1, arithmetic-intensity / accelerator-intensity)

[reference](https://jax-ml.github.io/scaling-book/roofline/)


<img src="images/deep-network.png" width="800" />

考查一个具有 $L$ 层，且输入、输出及中间激活值均为 $D$ 维的深度 MLP 网络模型。


In [ ]:
class Block(nn.Module):
    """Simple block that applies a linear transformation followed by a ReLU nonlinearity."""
    def __init__(self, dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dim, dim) / math.sqrt(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x @ self.weight  # 线性变换
        x = F.relu(x)        # 激活函数
        return x


In [ ]:
class DeepNetwork(nn.Module):
    """Map `dim`-vector to a `dim`-vector."""
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([Block(dim) for i in range(num_layers)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 顺次应用所有层
        for layer in self.layers:
            x = layer(x)
        return x


In [ ]:
D = 8  # 输入、激活和输出的维度
L = 3  # 网络层数
model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())
num_parameters = get_num_parameters(model)
assert num_parameters == (D * D) * L
B = 4  # 批大小
x = torch.randn(B, D, device=cuda_if_available())
y = model(x)


到目前为止，我们已经创建了各种张量并让它们执行前向传播。

现在，我们要开始计算梯度，即执行反向传播 (backward)。

作为一个极其简单的例子，让我们考查一个一维的线性模型：

$$y = 0.5 (x \cdot w - 5)^2$$

前向传播：计算 Loss


In [ ]:
x = torch.tensor([1., 2, 3])
w = torch.tensor([1., 1, 1], requires_grad=True)  # Want gradient
pred_y = x @ w
loss = 0.5 * (pred_y - 5).pow(2)


反向传播：计算梯度


In [ ]:
loss.backward()
assert torch.equal(w.grad, torch.tensor([1, 2, 3]))


接下来，我们来精确计算求解梯度所需的 FLOPs 开销。

<img src="images/deep-network.png" width="800" />


In [ ]:
B = 1024  # 样本数量
D = 256   # Dimension


定义一个简化的 2 层线性网络模型：


In [ ]:
x = torch.ones(B, D, device=cuda_if_available())
w1 = torch.randn(D, D, device=cuda_if_available(), requires_grad=True)
w2 = torch.randn(D, D, device=cuda_if_available(), requires_grad=True)
h1 = einsum(x, w1, "batch in, in out -> batch out")  # x @ w1
h2 = einsum(h1, w2, "batch in, in out -> batch out")  # h1 @ w2
loss = (h2.mean() - 0)**2  # Regress everything to 0 (arbitrary)
h1.retain_grad()  # 仅供调试检查使用
h2.retain_grad()  # 仅供调试检查使用
loss.backward()


## 聚焦于单个层级

我们重点分析第二层：$h_2 = h_1 W_2$

**前向传播**：回想一下前向矩阵乘法的 FLOPs 数量：


In [ ]:
num_forward_flops = 2 * B * D * D


**反向传播**：执行反向梯度计算需要多少 FLOPs？

我们需要计算：

- 相对输入激活值的梯度 `h1.grad`（$\frac{\partial L}{\partial h_1}$）

- 相对权重参数的梯度 `w2.grad`（$\frac{\partial L}{\partial W_2}$）


In [ ]:
h1_grad = einsum(h2.grad, w2, "batch out, in out -> batch in")
assert torch.allclose(h1.grad, h1_grad)
w2_grad = einsum(h2.grad, h1, "batch out, batch in -> in out")
assert torch.allclose(w2.grad, w2_grad)
num_backward_flops = (2 * B * D * D) + (2 * B * D * D)


我们非常清楚地看到，单个层的反向传播计算开销恰好是前向传播的 2 倍。

## 考虑网络的所有层

上面只考查了 $W_2$，反向传播必须贯穿整个神经网络的所有参数。

总结前向与反向的算力配比：

- **前向传播**：$2 \times \text{数据样本数} \times \text{模型参数量}$ FLOPs

- **反向传播**：$4 \times \text{数据样本数} \times \text{模型参数量}$ FLOPs

- **单步训练总计**：$6 \times \text{数据样本数} \times \text{模型参数量}$ FLOPs

这个极其著名的 “6 倍参数量” 经验法则同样也对多层感知机（MLP）和 Transformer 在短上下文下非常适用。

……并且它也给后面的作业中预估大规模预训练算力开销提供了最核心的理论依据。


回顾我们刚刚定义的深度神经网络。


In [ ]:
B = 2  # 批大小
D = 4  # 输入、激活和输出的维度
L = 3  # 网络层数
model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())


让我们定义一个 AdaGrad 优化器（作为实现定制优化器的展示）：

- 动量法 (Momentum) = SGD + 梯度的一阶指数移动平均

- AdaGrad = SGD + 累加梯度历史平方和进行自适应缩放

- RMSProp = AdaGrad + 梯度的二阶指数移动平均

- Adam = RMSProp + Momentum（结合一阶与二阶动量，当下大模型最常用的优化器）

AdaGrad 论文参考：[Adaptive Subgradient Methods for Online Learning and Stochastic Optimization (Duchi et al., 2011)](https://www.jmlr.org/papers/volume12/duchi11a/duchi11a.pdf)


In [ ]:
class AdaGrad(torch.optim.Optimizer):
    def __init__(self, params: Iterable[nn.Parameter], lr: float = 0.01):
        super(AdaGrad, self).__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                # 优化器状态
                state = self.state[p]
                grad = p.grad.data

                # 获取平方梯度累积值 g2 = sum_{i<t} g_i^2
                g2 = state.get("g2", torch.zeros_like(grad))

                # 更新优化器内部状态
                g2 += torch.square(grad)
                state["g2"] = g2

                # 更新模型参数
                p.data -= lr * grad / torch.sqrt(g2 + 1e-5)


In [ ]:
optimizer = AdaGrad(model.parameters(), lr=0.01)
state = model.state_dict()
x = torch.randn(B, D, device=cuda_if_available())
y = torch.tensor([4., 5.], device=cuda_if_available())
pred_y = model(x).mean()
loss = F.mse_loss(input=pred_y, target=y)
loss.backward()
optimizer.step()
optimizer_state = {i: dict(p_state) for i, (p, p_state) in enumerate(optimizer.state.items())}
optimizer.zero_grad(set_to_none=True)


## 显存核算 (Memory Accounting)


In [ ]:
num_parameters = D * D * L
parameter_memory = 2 * num_parameters  # (2 bytes for bf16)
gradient_memory = 2 * num_parameters  # (2 bytes for bf16)
optimizer_state_memory = 4 * num_parameters  # (4 bytes for fp32)
activation_memory = 2 * (B * D * L)  # (2 bytes for bf16)


为了数值更新稳定性，优化器的状态（如平方梯度和、动量）通常必须使用高精度 (fp32) 存储。

优化器状态的显存占用：

- AdaGrad：每个参数 4 字节（仅需存储二阶动量值 $g^2$）

- Adam：每个参数 8 字节（需要存储一阶动量 $m$ 和二阶动量 $v$）


In [ ]:
total_memory = parameter_memory + activation_memory + gradient_memory + optimizer_state_memory


## 单步训练计算开销


In [ ]:
num_parameters = D * D * L
flops = 6 * B * num_parameters


## 在 Transformer 中的资源核算

Transformer 里的资源账本核算要复杂一些（需要考虑多头注意力、KV 缓存等），但核心方法完全一致。

作业 1 将要求你亲手完成 Transformer 的资源核算。

有关 Transformer 训练显存分析的优秀博客： [相关文章](https://erees.dev/transformer-memory/)

有关 Transformer 训练算力 FLOPs 计算的优秀博客： [相关文章](https://www.adamcasson.com/posts/transformer-flops)


In [ ]:
D = 16  # Dimensionality
true_w = torch.arange(D, dtype=torch.float32, device=cuda_if_available())
B = 4  # 批大小
def get_batch() -> tuple[torch.Tensor, torch.Tensor]:
    x = torch.randn(B, D).to(cuda_if_available())
    true_y = x @ true_w
    return (x, true_y)
L = 2  # 网络层数
model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())
optimizer = AdaGrad(model.parameters(), lr=0.01)
num_train_steps = 3
for t in range(num_train_steps):
    # 提取数据 batch
    x, y = get_batch()

    # Forward (compute loss)
    pred_y = model(x).mean()
    loss = F.mse_loss(pred_y, y)

    # Backward (compute gradients)
    loss.backward()

    # 更新模型参数
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)


大批大小（Large batch sizes）能够平滑梯度，提升分布式训练的稳定性。

然而，前向中间激活值的显存随 Batch size 线性增加，很容易导致显存溢出（OOM）。


In [ ]:
B = 64     # 批大小
D = 1024   # Dimensionality
L = 16     # 网络层数
activation_memory = 2 * B * D * L  # (2 bytes for bf16)


**梯度累加 (Gradient Accumulation)** 的工作原理：

- 在更小的 Micro batch 上分别执行前向和反向，但不执行权重更新；

- 将梯度累加在参数的 `.grad` 中；

- 每隔若干步后（当有效 batch size 到达预期目标），让优化器步进更新一次权重，随后清空梯度。


In [ ]:
micro_batch_size = B / 4
activation_memory = 2 * micro_batch_size * D * L  # (2 bytes for bf16)


在进行模型训练时，反向传播必须使用前向的所有激活值以求解梯度，因而需将其保留在显存中。

而在模型推理时不需要求解梯度，因而前向之后可以直接释放前面的激活值，仅保留当前层级的信息即可。

<img src="images/deep-network.png" width="800" />

此时的显存开销对比：


In [ ]:
B = 64     # 批大小
D = 1024   # Dimensionality
L = 16     # 网络层数
x = torch.randn(B, D, device=cuda_if_available(), requires_grad=True)
activation_memory = 2 * B * D * L
model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())
memory = get_max_memory_usage(lambda: model(x).sum().backward())


我们能进一步压缩前向激活值占用的显存吗？

**激活值检查点 (Activation Checkpointing)**（又称梯度检查点或重算机制）：

核心思想：

- 前向传播：只保存极少数“检查点”层（如每段 Block 的输入）的激活值，丢弃中间结果；

- 反向传播：当某层计算梯度需要中间激活值时，从最近的检查点层出发重新运行一次前向重算（Rematerialization），复原所需激活值。

其核心本质是以少量的“重算时间”来换取海量的“显存空间”。


In [ ]:
class DeepNetworkCheckpointed(nn.Module):
    """Same as DeepNetwork, but with activation checkpointing."""
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([Block(dim) for i in range(num_layers)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 顺次应用所有层
        for layer in self.layers:
            # 核心：仅在检查点处保留激活值，其余计算通过重算复原
            x = torch.utils.checkpoint.checkpoint(layer, x)
        return x


In [ ]:
model = DeepNetworkCheckpointed(dim=D, num_layers=L).to(cuda_if_available())
checkpointed_memory = get_max_memory_usage(lambda: model(x).sum().backward())


对于超深的网络（$L$ 很大），我们能继续减少激活显存吗？

我们应该以多大的频率（间隔）设置检查点？

- **保存所有层**：激活显存为 $O(L)$，无重算开销。

- **完全不保存**：激活显存为 $O(1)$，但重算时间复杂度退化至 $O(L^2)$。

- **每隔 $\sqrt{L}$ 层保存一次**：实现完美折中，激活显存降低至 $O(\sqrt{L})$，重算开销仅为 $O(L)$。

## 第二讲总结

- 深度学习底层的计算全部都是围绕张量进行的（参数、梯度、中间激活、优化器状态、训练数据）。

- **Einops** 库提供了一种更健壮、不易出错且更清晰的张量维度管理和变换方式。

- 每次梯度更新需要执行大约 **$6 	imes N 	imes D$** 次浮点运算（FLOPs）。

- 通过**算术强度与 Roofline 拓扑分析**，我们可以判断硬件当前的运行状态究竟是受限于显存带宽还是计算算力。

- 大型矩阵乘法往往是**算力受限 (Compute-bound)** 的；而逐元素（Element-wise）操作或矩阵-向量乘法往往是**显存带宽受限 (Memory-bound)** 的。

- 我们可以通过**梯度累加**和**激活值检查点**这两大常用机制，以微小的时间牺牲，成倍降低训练时的显存峰值要求，从而能够训练更大规模的模型。
